# BLIP Fine-Tuning for Text-to-Video Retrieval

This notebook integrates BLIP with the project MSVD video-text format, uses the standardized three-frame configuration, fine-tunes BLIP with an image-text contrastive objective, and evaluates text-to-video retrieval.

`SMOKE_TEST` intentionally remains `True` for a safe reproducibility run. Set it to `False` only when performing the full experiment.

In [ ]:
!pip install -q transformers accelerate decord pillow tqdm pandas


In [ ]:
import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForImageTextRetrieval
from decord import VideoReader, cpu

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; using local paths.')


In [ ]:
# Reproducible experiment configuration
SEED = 298
DATA_ROOT = '/content/drive/Shared drives/DATA 298A/DATA/MSVD'
OUTPUT_DIR = '/content/blip_finetuned_outputs'
MODEL_NAME = 'Salesforce/blip-itm-base-coco'

# Shared project preprocessing configuration
FRAME_COUNT = 3
FRAME_SAMPLING = 'equidistant'
IMAGE_SIZE = 224
TOP_K_FRAMES = 2
MAX_TEXT_LEN = 64

# Keep this True for the requested smoke test.
SMOKE_TEST = True
SMOKE_VIDEOS = 8
EPOCHS = 1
TRAIN_BATCH_SIZE = 4
EVAL_TEXT_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0
# Evaluate all captions so BLIP uses the same query population as CLIP.
QUERY_POLICY = 'all captions per video'

os.makedirs(OUTPUT_DIR, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('SMOKE_TEST:', SMOKE_TEST)


## 1. Shared MSVD loader and standardized frame sampling

The loader follows the project shared split schema: `msvd_train.json`, `msvd_val.json`, and `msvd_test.json`, with videos stored in `raw_videos`. It groups captions by video ID so the train/validation/test split remains video-disjoint. If the team has a Python shared-loader module, replace the body of `load_shared_split` with that project import; the remainder of the notebook remains unchanged.

In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def clean_caption(text):
    return ' '.join(str(text).strip().split())

def make_video_index(video_root):
    index = {}
    for path in Path(video_root).iterdir():
        if path.is_file():
            index[path.name] = str(path)
            index[path.stem] = str(path)
    return index

def load_shared_split(split, max_videos=None):
    annotation_path = os.path.join(DATA_ROOT, f'msvd_{split}.json')
    video_root = os.path.join(DATA_ROOT, 'raw_videos')
    if not os.path.exists(annotation_path):
        raise FileNotFoundError(annotation_path)
    if not os.path.isdir(video_root):
        raise FileNotFoundError(video_root)

    video_index = make_video_index(video_root)
    by_video = {}
    for item in load_json(annotation_path):
        video_name = str(item.get('video', item.get('video_name', '')))
        video_path = video_index.get(video_name) or video_index.get(Path(video_name).stem)
        if not video_path:
            continue
        video_id = str(item.get('video_id', Path(video_name).stem))
        raw = item.get('caption', item.get('captions', []))
        captions = [raw] if isinstance(raw, str) else raw
        captions = [clean_caption(x) for x in captions if clean_caption(x)]
        if video_id not in by_video:
            by_video[video_id] = {'video_id': video_id, 'video_name': video_name, 'video_path': video_path, 'captions': []}
        by_video[video_id]['captions'].extend(captions)

    videos = list(by_video.values())
    for video in videos:
        video['captions'] = list(dict.fromkeys(video['captions']))
    videos.sort(key=lambda x: x['video_id'])
    return videos[:max_videos] if max_videos else videos

def sample_indices(total_frames, frame_count=FRAME_COUNT):
    if total_frames <= 0:
        raise ValueError('Video has no frames')
    return np.linspace(0, total_frames - 1, min(frame_count, total_frames), dtype=int).tolist()

def read_sampled_frames(video_path):
    reader = VideoReader(video_path, ctx=cpu(0))
    indices = sample_indices(len(reader), FRAME_COUNT)
    return [Image.fromarray(x).convert('RGB') for x in reader.get_batch(indices).asnumpy()]

limit = SMOKE_VIDEOS if SMOKE_TEST else None
train_videos = load_shared_split('train', limit)
val_videos = load_shared_split('val', limit)
test_videos = load_shared_split('test', limit)
print({'train': len(train_videos), 'val': len(val_videos), 'test': len(test_videos)})
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in val_videos)
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in test_videos)
assert set(x['video_id'] for x in val_videos).isdisjoint(x['video_id'] for x in test_videos)


## 2. BLIP-compatible dataset and smoke test

Each training item contains one standardized video frame and one caption. The smoke test confirms that the processor produces valid `pixel_values`, `input_ids`, and `attention_mask` tensors.

In [ ]:
class FrameCaptionDataset(Dataset):
    def __init__(self, videos, processor, max_items=None):
        self.processor = processor
        self.items = []
        for video in videos:
            for caption in video['captions']:
                for frame_index in range(FRAME_COUNT):
                    self.items.append((video, caption, frame_index))
        if max_items is not None:
            self.items = self.items[:max_items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        video, caption, frame_index = self.items[index]
        reader = VideoReader(video['video_path'], ctx=cpu(0))
        frame_id = sample_indices(len(reader), FRAME_COUNT)[frame_index]
        frame = Image.fromarray(reader[frame_id].asnumpy()).convert('RGB')
        encoded = self.processor(images=frame, text=caption, padding='max_length', truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
        return {key: value.squeeze(0) for key, value in encoded.items()}

processor = BlipProcessor.from_pretrained(MODEL_NAME)
smoke_dataset = FrameCaptionDataset(train_videos, processor, max_items=min(4, len(train_videos) * FRAME_COUNT))
smoke_loader = DataLoader(smoke_dataset, batch_size=min(2, len(smoke_dataset)), num_workers=NUM_WORKERS)
smoke_batch = next(iter(smoke_loader))
print({key: tuple(value.shape) for key, value in smoke_batch.items()})
assert smoke_batch['pixel_values'].ndim == 4
assert smoke_batch['input_ids'].ndim == 2
assert smoke_batch['input_ids'].shape[-1] == MAX_TEXT_LEN
assert 'attention_mask' in smoke_batch
print('SMOKE TEST PASSED: BLIP-compatible tensors were created.')


## 3. Fine-tune BLIP

The model is trained with a symmetric in-batch image-text contrastive loss. Checkpoints and training history are saved after every epoch.

In [ ]:
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
train_dataset = FrameCaptionDataset(train_videos, processor, max_items=32 if SMOKE_TEST else None)
val_dataset = FrameCaptionDataset(val_videos, processor, max_items=16 if SMOKE_TEST else None)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def move_batch(batch):
    return {key: value.to(DEVICE) for key, value in batch.items() if key in {'pixel_values', 'input_ids', 'attention_mask'}}

def contrastive_loss(batch):
    image_out = model.vision_model(pixel_values=batch['pixel_values'], return_dict=True)
    image_features = F.normalize(model.vision_proj(image_out.last_hidden_state[:, 0, :]), dim=-1)
    text_out = model.text_encoder(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], return_dict=True)
    text_features = F.normalize(model.text_proj(text_out.last_hidden_state[:, 0, :]), dim=-1)
    logits = image_features @ text_features.T
    labels = torch.arange(logits.size(0), device=DEVICE)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

def run_epoch(loader, training):
    model.train(training)
    total, count = 0.0, 0
    for batch in tqdm(loader, leave=False, desc='train' if training else 'validation'):
        batch = move_batch(batch)
        with torch.set_grad_enabled(training):
            loss = contrastive_loss(batch)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total += loss.item() * batch['input_ids'].size(0)
        count += batch['input_ids'].size(0)
    return total / max(count, 1)

history, best_val = [], float('inf')
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss}
    history.append(row)
    print(row)
    epoch_dir = os.path.join(OUTPUT_DIR, f'checkpoint-epoch-{epoch}')
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))
        processor.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))

with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print('Saved training evidence to', OUTPUT_DIR)


## 4. Retrieval evaluation

Every available caption is evaluated as a separate text query. The caption's associated video is its ground-truth target, matching the CLIP evaluation protocol. The video score is the mean of the two strongest frame similarities.

In [ ]:
@torch.inference_mode()
def encode_image_frames(model, processor, images):
    inputs = processor(images=images, return_tensors='pt')
    outputs = model.vision_model(pixel_values=inputs['pixel_values'].to(DEVICE), return_dict=True)
    features = model.vision_proj(outputs.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def encode_text_queries(model, processor, texts):
    inputs = processor(text=texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
    outputs = model.text_encoder(input_ids=inputs['input_ids'].to(DEVICE), attention_mask=inputs['attention_mask'].to(DEVICE), return_dict=True)
    features = model.text_proj(outputs.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def evaluate_retrieval(model, processor, videos):
    model.eval()
    video_frames, video_ids, queries = [], [], []
    for video in tqdm(videos, desc='Encoding videos'):
        video_frames.append(encode_image_frames(model, processor, read_sampled_frames(video['video_path'])))
        video_ids.append(video['video_id'])
        # Add every caption so BLIP and CLIP use the same evaluation population.
        # A full MSVD test run should contain roughly 27,000 queries, rather
        # than only one query for each of approximately 670 videos.
        for caption in video['captions']:
            queries.append({'video_id': video['video_id'], 'caption': caption})

    text_chunks = []
    for start in range(0, len(queries), EVAL_TEXT_BATCH_SIZE):
        text_chunks.append(encode_text_queries(model, processor, [q['caption'] for q in queries[start:start + EVAL_TEXT_BATCH_SIZE]]))
    text_embeddings = torch.cat(text_chunks)

    scores = np.zeros((len(queries), len(videos)), dtype=np.float32)
    for video_index, frames in enumerate(video_frames):
        frame_scores = text_embeddings @ frames.T
        scores[:, video_index] = torch.topk(frame_scores, k=min(TOP_K_FRAMES, frames.shape[0]), dim=1).values.mean(dim=1).numpy()

    id_to_index = {video_id: index for index, video_id in enumerate(video_ids)}
    ranks = []
    for query_index, query in enumerate(queries):
        ranking = np.argsort(-scores[query_index])
        ranks.append(int(np.where(ranking == id_to_index[query['video_id']])[0][0]) + 1)
    ranks = np.asarray(ranks)
    return {
        'R@1': float(np.mean(ranks <= 1)),
        'R@5': float(np.mean(ranks <= 5)),
        'R@10': float(np.mean(ranks <= 10)),
        'MRR': float(np.mean(1.0 / ranks)),
        'MeanRank': float(np.mean(ranks)),
        'MedianRank': float(np.median(ranks)),
        'num_queries': int(len(ranks)),
        'num_videos': int(len(videos))
    }

best_model = BlipForImageTextRetrieval.from_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint')).to(DEVICE)
best_processor = BlipProcessor.from_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))
fine_tuned_metrics = evaluate_retrieval(best_model, best_processor, test_videos)
print(pd.DataFrame([fine_tuned_metrics]))
with open(os.path.join(OUTPUT_DIR, 'fine_tuned_blip_metrics.json'), 'w') as f:
    json.dump(fine_tuned_metrics, f, indent=2)


## 5. Zero-shot comparison

The zero-shot model uses the same test videos, frame sampling, pooling, query policy, and metrics as the fine-tuned model.

In [ ]:
zero_shot_model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
zero_shot_metrics = evaluate_retrieval(zero_shot_model, processor, test_videos)
with open(os.path.join(OUTPUT_DIR, 'zero_shot_blip_metrics.json'), 'w') as f:
    json.dump(zero_shot_metrics, f, indent=2)

metric_names = ['R@1', 'R@5', 'R@10', 'MRR', 'MeanRank', 'MedianRank']
comparison = pd.DataFrame({
    'Metric': metric_names,
    'Zero-shot BLIP': [zero_shot_metrics[x] for x in metric_names],
    'Fine-tuned BLIP': [fine_tuned_metrics[x] for x in metric_names]
})
comparison['Change'] = comparison['Fine-tuned BLIP'] - comparison['Zero-shot BLIP']
display(comparison)

configuration = {
    'dataset': 'MSVD',
    'model': MODEL_NAME,
    'frame_count': FRAME_COUNT,
    'frame_sampling': FRAME_SAMPLING,
    'image_size': IMAGE_SIZE,
    'top_k_frames': TOP_K_FRAMES,
    'query_policy': QUERY_POLICY,
    'smoke_test': SMOKE_TEST,
    'smoke_videos': SMOKE_VIDEOS,
    'epochs': EPOCHS,
    'batch_size': TRAIN_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'max_text_length': MAX_TEXT_LEN,
    'seed': SEED
}
with open(os.path.join(OUTPUT_DIR, 'blip_zero_shot_vs_fine_tuned.json'), 'w') as f:
    json.dump({'zero_shot': zero_shot_metrics, 'fine_tuned': fine_tuned_metrics, 'configuration': configuration}, f, indent=2)


## Evidence checklist

This smoke run should produce:

- `SMOKE TEST PASSED` output with tensor shapes
- `training_history.json`
- `checkpoint-epoch-1/` and `best_checkpoint/`
- `fine_tuned_blip_metrics.json`
- `zero_shot_blip_metrics.json`
- `blip_zero_shot_vs_fine_tuned.json`

Because `SMOKE_TEST = True`, this run is integration evidence rather than final performance evidence. For final project metrics, rerun the same notebook with `SMOKE_TEST = False` and retain the resulting checkpoint and JSON files.